# Docile eda

In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()

while not (project_root / "app").is_dir():
    if project_root == project_root.parent:
        raise RuntimeError("Cannot find the project root containing app/")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import pandas as pd

In [3]:
df = pd.read_parquet('../docile_df.parquet')
df

,document_id,ocr_text,text_with_boxes,json_data,pdf_path
0,00134dd365a24343b35b78c6,"WIRE INSTRUCTIONS: REMIT TO: NationsBank, N.A....","[{'bbox': [0.5254137115839244, 0.1701559020044...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/00134dd365a24343b35b78c6.pdf
1,00136a27c7774c1e8dc6b2f2,Please Remit To: KMOZ 92.3 The Moose 1360 E. S...,"[{'bbox': [0.4258823529411765, 0.0481818181818...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/00136a27c7774c1e8dc6b2f2.pdf
2,002e3cf97973428f905671b3,"UMI PUBLICATIONS, INC. o E. P.O.BOX 30036 (282...","[{'bbox': [0.16857314870559903, 0.031982942430...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/002e3cf97973428f905671b3.pdf
3,002f9b82b74f4258b3b072d0,Page 1 of1 INVOICE WZZM WZZM 645: 3 Mile Road ...,"[{'bbox': [0.9294117647058824, 0.0159090909090...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/002f9b82b74f4258b3b072d0.pdf
4,003568b1286f4dab953fc2d5,10032089 Appud WAIT KLEN SAINISIC INVOICE Walt...,"[{'bbox': [0.7651651651651652, 0.1286874732791...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/003568b1286f4dab953fc2d5.pdf
...,...,...,...,...,...
5675,ffcaffe270654a87ad5a0631,Page 1of 2 INVOICE Property WPHI-FM WPHI-FM In...,"[{'bbox': [0.8590381426202321, 0.0389055151774...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/ffcaffe270654a87ad5a0631.pdf
5676,ffcc5e0860f54314858dae44,"Invoice / Affidavit OHANA MEDIA GROUP, LLC. We...","[{'bbox': [0.39941176470588236, 0.052727272727...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/ffcc5e0860f54314858dae44.pdf
5677,ffcd395d52ff4a83bf659333,OCT 242001 U.S.A. Purchase Requisition Requisi...,"[{'bbox': [0.6539365452408931, 0.0840527033166...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/ffcd395d52ff4a83bf659333.pdf
5678,ffd8e8fad20b4cc5984b3f26,"UNIVERSITY OF CALIFORNIA, DAVIS Accounting Off...","[{'bbox': [0.33274231678487, 0.085523385300668...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/ffd8e8fad20b4cc5984b3f26.pdf


In [5]:
audit = pd.read_json(
    "../data/interim/docile_audit/documents.jsonl",
    lines=True,
)

reviewed = df.merge(
    audit[["document_id", "training_candidate", "review_reason"]],
    on="document_id",
    validate="one_to_one",
)

legit_df = reviewed.loc[reviewed["training_candidate"]].copy()
review_df = reviewed.loc[~reviewed["training_candidate"]].copy()

In [6]:
reviewed["review_reason"].value_counts()

review_reason
accepted                              5403
annotation_box_without_word_center     256
remaining_label_conflict                14
no_label_coverage_after_priority         7
Name: count, dtype: int64

In [7]:
reviewed

,document_id,ocr_text,text_with_boxes,json_data,pdf_path,training_candidate,review_reason
0,00134dd365a24343b35b78c6,"WIRE INSTRUCTIONS: REMIT TO: NationsBank, N.A....","[{'bbox': [0.5254137115839244, 0.1701559020044...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/00134dd365a24343b35b78c6.pdf,True,accepted
1,00136a27c7774c1e8dc6b2f2,Please Remit To: KMOZ 92.3 The Moose 1360 E. S...,"[{'bbox': [0.4258823529411765, 0.0481818181818...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/00136a27c7774c1e8dc6b2f2.pdf,True,accepted
2,002e3cf97973428f905671b3,"UMI PUBLICATIONS, INC. o E. P.O.BOX 30036 (282...","[{'bbox': [0.16857314870559903, 0.031982942430...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/002e3cf97973428f905671b3.pdf,True,accepted
3,002f9b82b74f4258b3b072d0,Page 1 of1 INVOICE WZZM WZZM 645: 3 Mile Road ...,"[{'bbox': [0.9294117647058824, 0.0159090909090...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/002f9b82b74f4258b3b072d0.pdf,True,accepted
4,003568b1286f4dab953fc2d5,10032089 Appud WAIT KLEN SAINISIC INVOICE Walt...,"[{'bbox': [0.7651651651651652, 0.1286874732791...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/003568b1286f4dab953fc2d5.pdf,True,accepted
...,...,...,...,...,...,...,...
5675,ffcaffe270654a87ad5a0631,Page 1of 2 INVOICE Property WPHI-FM WPHI-FM In...,"[{'bbox': [0.8590381426202321, 0.0389055151774...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/ffcaffe270654a87ad5a0631.pdf,True,accepted
5676,ffcc5e0860f54314858dae44,"Invoice / Affidavit OHANA MEDIA GROUP, LLC. We...","[{'bbox': [0.39941176470588236, 0.052727272727...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/ffcc5e0860f54314858dae44.pdf,True,accepted
5677,ffcd395d52ff4a83bf659333,OCT 242001 U.S.A. Purchase Requisition Requisi...,"[{'bbox': [0.6539365452408931, 0.0840527033166...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/ffcd395d52ff4a83bf659333.pdf,True,accepted
5678,ffd8e8fad20b4cc5984b3f26,"UNIVERSITY OF CALIFORNIA, DAVIS Accounting Off...","[{'bbox': [0.33274231678487, 0.085523385300668...",{'invoice': {'client_address': {'bbox': [array...,/content/docile/pdfs/ffd8e8fad20b4cc5984b3f26.pdf,True,accepted


In [11]:
import json
from pathlib import Path

path = Path("../data/interim/docile_audit/summary.json")

with path.open(encoding="utf-8") as file:
    summary = json.load(file)

summary_df = (
    pd.DataFrame.from_dict(summary["field_counts"], orient="index")
    .rename_axis("field")
    .reset_index()
)

In [12]:
summary_df

,field,documents,occurrences,matched,unmatched,exact,ratio80,fragmented
0,invoice.client_address,4743,5249,5249,0,3192,5215,4039
1,invoice.client_name,5488,6142,6142,0,5103,6082,858
2,invoice.due_date,742,884,881,3,817,860,1
3,invoice.invoice_date,5013,6214,6077,137,5563,5891,9
4,invoice.invoice_number,4711,6141,6081,60,5696,5986,1
5,invoice.seller_address,4577,6634,6633,1,3678,6552,3698
6,invoice.seller_name,4968,7354,7348,6,6021,7250,527
7,items.amount_net,805,4235,3726,509,3571,3690,2
8,items.code,1052,8904,8754,150,7522,8456,267
9,items.currency,285,1196,1085,111,1167,1167,0


In [14]:
occurences_df = pd.read_json(
    "../data/interim/docile_audit/occurrences.jsonl",
    lines=True,
)
occurences_df

,document_id,field,item_id,occurrence,page,valid_bbox,matched_words,overlap50_words,overlap80_words,runs,normalized_exact,text_ratio,expected,observed
0,00134dd365a24343b35b78c6,invoice.client_address,NaN,0,0,True,13,13,13,3,False,97.727273,LORILLARD RESEARCH CENTER\nATTN: MS MELANEE BE...,LORILLARD RESEARCH CENTER ATTN: MS MELANEE BEN...
1,00134dd365a24343b35b78c6,invoice.client_name,NaN,0,0,True,3,3,3,1,True,100.000000,LORILLARD RESEARCH CENTER,LORILLARD RESEARCH CENTER
2,00134dd365a24343b35b78c6,invoice.invoice_date,NaN,0,0,True,1,1,1,1,False,87.500000,03/25/99,03/25799
3,00134dd365a24343b35b78c6,invoice.invoice_number,NaN,0,0,True,1,1,1,1,True,100.000000,990304502,990304502
4,00134dd365a24343b35b78c6,invoice.seller_address,NaN,0,0,True,13,13,13,2,True,100.000000,"BIORELIANCE- Testing & Development, Inc.\n1492...","BIORELIANCE Testing & Development, Inc. 14920 ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
228977,ffeeeefed0b24da28a16c8a4,items.date,9.0,1,1,True,1,1,1,1,True,100.000000,11/7/2019,11/7/2019
228978,ffeeeefed0b24da28a16c8a4,items.position,9.0,0,1,True,1,1,1,1,True,100.000000,1,1
228979,ffeeeefed0b24da28a16c8a4,items.quantity,9.0,0,1,True,1,1,1,1,True,100.000000,7,7
228980,ffeeeefed0b24da28a16c8a4,items.total_price,9.0,0,1,True,1,1,1,1,True,100.000000,$385,$385


In [19]:
row = df.iloc[1]
ocr_text = row['ocr_text']
ocr_text

'Please Remit To: KMOZ 92.3 The Moose 1360 E. Sherwood Drive Grand Junction, CO 81501 970.254.2100 ROW CDU Invoice Date 17685-2 10/31/2020 Invoice Total: $806.00 Mesa County Republicans Payment Terms: 30 Days 920 Visa Ct. Invoice Month: October Grand Junction, CO 81506 Advertiser: Mesa County Republicans Account: Mesa County Republicans Billing Type: Monthly DETACH AND RETURN WITH PAYMENT Advertiser: Mesa County Republicans Salesperson: Brenda Flight Dates: 10/1/2020 10/31/2020 Description: Vote Republican Est. Number: IDB Code: Date Length KMOZ-FM Qty Rate To Spot Classification: CM KMOZ-FM Copy Name: MESA CO REPUBLICAN ISCI: ASNGBPEMBAEAEPCN PARTY/2020-Vote Republican 10/1/2020 30 5:21:31 PM 26.00 26.00 10/2/2020 30 3:57:34 PM 1 26.00 26.00 10/3/2020 30 4:45:08 PM 1 26.00 26.00 10/4/2020 30 6:42:21 PM 1 26.00 26.00 10/5/2020 30 6:51:05 PM 1 26.00 26.00 10/6/2020 30 10:14:36 AM 1 26.00 26.00 10/7/2020 30 5:38:42 PM 1 26.00 26.00 10/8/2020 30 6:23:24 PM 1 26.00 26.00 10/9/2020 30 12:49

In [21]:
json_data = row['json_data']
json_data

{'invoice': {'client_address': {'bbox': array([array([0.04376009, 0.17086981, 0.23980532, 0.21618521]),
          array([0.0472609 , 0.17111639, 0.23893011, 0.2150791 ])],
         dtype=object),
   'pages': array([1, 0]),
   'source_values': array(['Mesa County Republicans\n920 Visa Ct.\nGrand Junction, CO 81506',
          'Mesa County Republicans\n920 Visa Ct.\nGrand Junction, CO 81506'],
         dtype=object),
   'value': 'Mesa County Republicans\n920 Visa Ct.\nGrand Junction, CO 81506'},
  'client_name': {'bbox': array([array([0.04516129, 0.17190547, 0.24068052, 0.18678515]),
          array([0.04516129, 0.17154615, 0.23805491, 0.18574949])],
         dtype=object),
   'pages': array([0, 1]),
   'source_values': array(['Mesa County Republicans', 'Mesa County Republicans'], dtype=object),
   'value': 'Mesa County Republicans'},
  'due_date': None,
  'invoice_date': {'bbox': array([array([0.79643372, 0.14000432, 0.86693548, 0.15217861]),
          array([0.79730892, 0.13975774, 0.8

In [25]:
json_data.keys()

dict_keys(['invoice', 'items', 'line_item_headers', 'metadata', 'other_fields', 'payment_instructions', 'subtotal'])

In [26]:
json_data['invoice'].keys()

dict_keys(['client_address', 'client_name', 'due_date', 'invoice_date', 'invoice_number', 'seller_address', 'seller_name'])

In [29]:
json_data['items'][0].keys()

dict_keys(['fields', 'line_item_id'])

In [30]:
json_data['items'][0]['fields'].keys()

dict_keys(['amount_net', 'code', 'currency', 'date', 'description', 'discount_amount', 'discount_rate', 'hts_number', 'order_id', 'person_name', 'position', 'quantity', 'tax', 'tax_rate', 'total_price', 'unit_price', 'unit_price_net', 'units_of_measure', 'weight'])

In [32]:
json_data['items'][0]['line_item_id']

1

In [33]:
json_data['payment_instructions'].keys()

dict_keys(['account_number', 'payment_terms'])

In [34]:
json_data['subtotal'].keys()

dict_keys(['tax', 'total'])

In [35]:
json_data['metadata'].keys()

dict_keys(['cluster_id', 'currency', 'document_type', 'language', 'original_filename', 'page_count', 'page_sizes_at_200dpi', 'page_to_table_grid', 'source'])

In [36]:
json_data['other_fields'].keys()

dict_keys(['amount_due', 'amount_paid', 'amount_total_net', 'bank_num', 'bic', 'currency_code_amount_due', 'customer_delivery_address', 'customer_delivery_name', 'customer_id', 'customer_order_id', 'customer_other_address', 'customer_other_name', 'customer_registration_id', 'customer_tax_id', 'iban', 'order_id', 'payment_reference', 'tax_detail_gross', 'tax_detail_net', 'tax_detail_rate', 'tax_detail_tax', 'vendor_email', 'vendor_order_id', 'vendor_registration_id', 'vendor_tax_id'])

In [38]:
invoice_section = json_data['invoice']
invoice_section

{'client_address': {'bbox': array([array([0.04376009, 0.17086981, 0.23980532, 0.21618521]),
         array([0.0472609 , 0.17111639, 0.23893011, 0.2150791 ])],
        dtype=object),
  'pages': array([1, 0]),
  'source_values': array(['Mesa County Republicans\n920 Visa Ct.\nGrand Junction, CO 81506',
         'Mesa County Republicans\n920 Visa Ct.\nGrand Junction, CO 81506'],
        dtype=object),
  'value': 'Mesa County Republicans\n920 Visa Ct.\nGrand Junction, CO 81506'},
 'client_name': {'bbox': array([array([0.04516129, 0.17190547, 0.24068052, 0.18678515]),
         array([0.04516129, 0.17154615, 0.23805491, 0.18574949])],
        dtype=object),
  'pages': array([0, 1]),
  'source_values': array(['Mesa County Republicans', 'Mesa County Republicans'], dtype=object),
  'value': 'Mesa County Republicans'},
 'due_date': None,
 'invoice_date': {'bbox': array([array([0.79643372, 0.14000432, 0.86693548, 0.15217861]),
         array([0.79730892, 0.13975774, 0.86693548, 0.15125568])],
    

In [42]:
client_address = invoice_section['client_address']
client_address

{'bbox': array([array([0.04376009, 0.17086981, 0.23980532, 0.21618521]),
        array([0.0472609 , 0.17111639, 0.23893011, 0.2150791 ])],
       dtype=object),
 'pages': array([1, 0]),
 'source_values': array(['Mesa County Republicans\n920 Visa Ct.\nGrand Junction, CO 81506',
        'Mesa County Republicans\n920 Visa Ct.\nGrand Junction, CO 81506'],
       dtype=object),
 'value': 'Mesa County Republicans\n920 Visa Ct.\nGrand Junction, CO 81506'}

In [44]:
text_with_boxes = row['text_with_boxes']
text_with_boxes

array([{'bbox': array([0.42588235, 0.04818182, 0.46705882, 0.05590909]), 'block_index': 0, 'confidence': 0.9999462366104126, 'end': 6, 'line_index': 0, 'page': 0, 'start': 0, 'text': 'Please', 'word_index': 0},
       {'bbox': array([0.47235294, 0.04818182, 0.50941176, 0.05590909]), 'block_index': 0, 'confidence': 0.9987379908561707, 'end': 12, 'line_index': 0, 'page': 0, 'start': 7, 'text': 'Remit', 'word_index': 1},
       {'bbox': array([0.51235294, 0.04818182, 0.53176471, 0.05590909]), 'block_index': 0, 'confidence': 0.9999043941497803, 'end': 16, 'line_index': 0, 'page': 0, 'start': 13, 'text': 'To:', 'word_index': 2},
       {'bbox': array([0.39294118, 0.06545455, 0.43941176, 0.075     ]), 'block_index': 0, 'confidence': 0.9999353885650635, 'end': 21, 'line_index': 1, 'page': 0, 'start': 17, 'text': 'KMOZ', 'word_index': 0},
       {'bbox': array([0.44470588, 0.06545455, 0.47588235, 0.075     ]), 'block_index': 0, 'confidence': 0.999804675579071, 'end': 26, 'line_index': 1, 'page

In [43]:
ca_bbox = client_address['bbox']
ca_pages = client_address['pages']

In [45]:
def word_inside_box(word, bbox, page):
    left, top, right, bottom = bbox
    word_left, word_top, word_right, word_bottom = word["bbox"]

    center_x = (word_left + word_right) / 2
    center_y = (word_top + word_bottom) / 2

    return (
        word["page"] == page
        and left <= center_x <= right
        and top <= center_y <= bottom
    )

In [46]:
text_with_boxes

array([{'bbox': array([0.42588235, 0.04818182, 0.46705882, 0.05590909]), 'block_index': 0, 'confidence': 0.9999462366104126, 'end': 6, 'line_index': 0, 'page': 0, 'start': 0, 'text': 'Please', 'word_index': 0},
       {'bbox': array([0.47235294, 0.04818182, 0.50941176, 0.05590909]), 'block_index': 0, 'confidence': 0.9987379908561707, 'end': 12, 'line_index': 0, 'page': 0, 'start': 7, 'text': 'Remit', 'word_index': 1},
       {'bbox': array([0.51235294, 0.04818182, 0.53176471, 0.05590909]), 'block_index': 0, 'confidence': 0.9999043941497803, 'end': 16, 'line_index': 0, 'page': 0, 'start': 13, 'text': 'To:', 'word_index': 2},
       {'bbox': array([0.39294118, 0.06545455, 0.43941176, 0.075     ]), 'block_index': 0, 'confidence': 0.9999353885650635, 'end': 21, 'line_index': 1, 'page': 0, 'start': 17, 'text': 'KMOZ', 'word_index': 0},
       {'bbox': array([0.44470588, 0.06545455, 0.47588235, 0.075     ]), 'block_index': 0, 'confidence': 0.999804675579071, 'end': 26, 'line_index': 1, 'page

In [47]:
ca_bbox, ca_pages

(array([array([0.04376009, 0.17086981, 0.23980532, 0.21618521]),
        array([0.0472609 , 0.17111639, 0.23893011, 0.2150791 ])],
       dtype=object),
 array([1, 0]))

In [52]:
spans = []

for occurrence, (bbox, page) in enumerate(
    zip(ca_bbox, ca_pages, strict=True)
):
    previous_index = None

    for index, word in enumerate(text_with_boxes):
        if not word_inside_box(word, bbox, page):
            continue

        # Merge only consecutive words in the original OCR sequence.
        if previous_index is not None and index == previous_index + 1:
            spans[-1]["end"] = int(word["end"])
        else:
            spans.append({
                "start": int(word["start"]),
                "end": int(word["end"]),
                "label": "invoice.client_address",
                "page": int(page),
                "occurrence": occurrence,
            })

        previous_index = index

spans.sort(key=lambda span: span["start"])

for span in spans:
    span["text"] = ocr_text[span["start"]:span["end"]]

In [53]:
spans

[{'start': 161,
  'end': 184,
  'label': 'invoice.client_address',
  'page': 0,
  'occurrence': 1,
  'text': 'Mesa County Republicans'},
 {'start': 208,
  'end': 220,
  'label': 'invoice.client_address',
  'page': 0,
  'occurrence': 1,
  'text': '920 Visa Ct.'},
 {'start': 244,
  'end': 268,
  'label': 'invoice.client_address',
  'page': 0,
  'occurrence': 1,
  'text': 'Grand Junction, CO 81506'},
 {'start': 1745,
  'end': 1768,
  'label': 'invoice.client_address',
  'page': 1,
  'occurrence': 0,
  'text': 'Mesa County Republicans'},
 {'start': 1792,
  'end': 1804,
  'label': 'invoice.client_address',
  'page': 1,
  'occurrence': 0,
  'text': '920 Visa Ct.'},
 {'start': 1828,
  'end': 1852,
  'label': 'invoice.client_address',
  'page': 1,
  'occurrence': 0,
  'text': 'Grand Junction, CO 81506'}]

In [55]:
for occurrence, expected in enumerate(client_address["source_values"]):
    observed = " ".join(
        span["text"]
        for span in spans
        if span["occurrence"] == occurrence
    )

    print("Expected:", expected.replace("\n", " "))
    print("Observed:", observed)
    print()

Expected: Mesa County Republicans 920 Visa Ct. Grand Junction, CO 81506
Observed: Mesa County Republicans 920 Visa Ct. Grand Junction, CO 81506

Expected: Mesa County Republicans 920 Visa Ct. Grand Junction, CO 81506
Observed: Mesa County Republicans 920 Visa Ct. Grand Junction, CO 81506



In [81]:
SECTIONS = ("invoice", "payment_instructions", "subtotal")


def find_spans(row):
    ocr_text = row["ocr_text"]
    text_with_boxes = row["text_with_boxes"]
    spans = []

    for section in SECTIONS:
        for name, field in row["json_data"].get(section, {}).items():
            if field is None:
                continue

            path = f"{section}.{name}"

            for occurrence, (bbox, page) in enumerate(
                zip(field["bbox"], field["pages"], strict=True)
            ):
                previous_index = None

                for word_index, word in enumerate(text_with_boxes):
                    if not word_inside_box(word, bbox, page):
                        continue

                    if (
                        previous_index is not None
                        and word_index == previous_index + 1
                    ):
                        spans[-1]["end"] = int(word["end"])
                    else:
                        spans.append({
                            "start": int(word["start"]),
                            "end": int(word["end"]),
                            "label": path,
                            "page": int(page),
                            "occurrence": occurrence,
                        })

                    previous_index = word_index

    spans.sort(key=lambda span: span["start"])

    for span in spans:
        span["text"] = ocr_text[span["start"]:span["end"]]

    return spans

In [84]:
spans = find_spans(row)
spans

[{'start': 17,
  'end': 84,
  'label': 'invoice.seller_address',
  'page': 0,
  'occurrence': 1,
  'text': 'KMOZ 92.3 The Moose 1360 E. Sherwood Drive Grand Junction, CO 81501'},
 {'start': 17,
  'end': 36,
  'label': 'invoice.seller_name',
  'page': 0,
  'occurrence': 0,
  'text': 'KMOZ 92.3 The Moose'},
 {'start': 119,
  'end': 126,
  'label': 'invoice.invoice_number',
  'page': 0,
  'occurrence': 0,
  'text': '17685-2'},
 {'start': 127,
  'end': 137,
  'label': 'invoice.invoice_date',
  'page': 0,
  'occurrence': 0,
  'text': '10/31/2020'},
 {'start': 153,
  'end': 160,
  'label': 'subtotal.total',
  'page': 0,
  'occurrence': 0,
  'text': '$806.00'},
 {'start': 161,
  'end': 184,
  'label': 'invoice.client_address',
  'page': 0,
  'occurrence': 1,
  'text': 'Mesa County Republicans'},
 {'start': 161,
  'end': 184,
  'label': 'invoice.client_name',
  'page': 0,
  'occurrence': 0,
  'text': 'Mesa County Republicans'},
 {'start': 200,
  'end': 207,
  'label': 'payment_instructions.pay

In [ ]:
from app.extraction.bert.normalization import Normalizer

normalizer = Normalizer()


def compare_spans(row, spans):
    comparisons = []

    for section in SECTIONS:
        for name, field in row["json_data"].get(section, {}).items():
            if field is None:
                continue

            path = f"{section}.{name}"

            for occurrence, (page, expected) in enumerate(
                zip(field["pages"], field["source_values"], strict=True)
            ):
                matching = [
                    span
                    for span in spans
                    if span["label"] == path
                    and span["occurrence"] == occurrence
                    and span["page"] == page
                ]

                observed = " ".join(span["text"] for span in matching)

                normalized_equal = (
                    normalizer.normalize(expected, path)
                        == normalizer.normalize(observed, path)
                )
                comparisons.append({
                    "field": path,
                    "occurrence": occurrence,
                    "page": int(page),
                    "expected": expected,
                    "observed": observed,
                    "span_count": len(matching),
                    "matched": bool(matching),
                    "normalized_equal": normalized_equal,
                })

    return pd.DataFrame(comparisons)

In [86]:
compare_df = compare_spans(row, spans)
compare_df

,field,occurrence,page,expected,observed,span_count,matched,normalized_equal
0,invoice.client_address,0,1,Mesa County Republicans\n920 Visa Ct.\nGrand J...,Mesa County Republicans 920 Visa Ct. Grand Jun...,3,True,True
1,invoice.client_address,1,0,Mesa County Republicans\n920 Visa Ct.\nGrand J...,Mesa County Republicans 920 Visa Ct. Grand Jun...,3,True,True
2,invoice.client_name,0,0,Mesa County Republicans,Mesa County Republicans,1,True,True
3,invoice.client_name,1,1,Mesa County Republicans,Mesa County Republicans,1,True,True
4,invoice.invoice_date,0,0,10/31/2020,10/31/2020,1,True,True
5,invoice.invoice_date,1,1,10/31/2020,10/31/2020,1,True,True
6,invoice.invoice_number,0,0,17685-2,17685-2,1,True,True
7,invoice.invoice_number,1,1,17685-2,17685-2,1,True,True
8,invoice.seller_address,0,1,KMOZ 92.3 The Moose\n1360 E. Sherwood Drive\nG...,KMOZ 92.3 The Moose 1360 E. Sherwood Drive Gra...,1,True,True
9,invoice.seller_address,1,0,KMOZ 92.3 The Moose\n1360 E. Sherwood Drive\nG...,KMOZ 92.3 The Moose 1360 E. Sherwood Drive Gra...,1,True,True


In [ ]:
total_span = next(
    span for span in spans
    if span["label"] == "subtotal.total"
    and span["occurrence"] == 1
)

display(pd.DataFrame([
    word
    for word in text_with_boxes
    if word["page"] == total_span["page"]
    and word["start"] < total_span["end"] + 20
    and word["end"] > total_span["start"] - 20
]))

,bbox,block_index,confidence,end,line_index,page,start,text,word_index
0,"[0.48705882352941177, 0.8295454545454546, 0.52...",15,0.993645,2523,0,1,2518,Spots,1
1,"[0.5741176470588235, 0.8295454545454546, 0.587...",16,0.999958,2526,0,1,2524,31,0
2,"[0.6211764705882353, 0.9059090909090909, 0.665...",17,0.777125,2534,0,1,2527,Invoice,0
3,"[0.6711764705882353, 0.9059090909090909, 0.701...",17,0.622064,2540,0,1,2535,Total,1
4,"[0.8723529411764706, 0.9059090909090909, 0.900...",18,0.999989,2546,0,1,2541,$806.,0


In [89]:
best_per_field = (
    compare_df
    .loc[compare_df["matched"]]
    .sort_values(
        ["field", "normalized_equal", "page", "occurrence"],
        ascending=[True, False, True, True],
    )
    .drop_duplicates("field")
    .reset_index(drop=True)
)
best_per_field


,field,occurrence,page,expected,observed,span_count,matched,normalized_equal
0,invoice.client_address,1,0,Mesa County Republicans\n920 Visa Ct.\nGrand J...,Mesa County Republicans 920 Visa Ct. Grand Jun...,3,True,True
1,invoice.client_name,0,0,Mesa County Republicans,Mesa County Republicans,1,True,True
2,invoice.invoice_date,0,0,10/31/2020,10/31/2020,1,True,True
3,invoice.invoice_number,0,0,17685-2,17685-2,1,True,True
4,invoice.seller_address,1,0,KMOZ 92.3 The Moose\n1360 E. Sherwood Drive\nG...,KMOZ 92.3 The Moose 1360 E. Sherwood Drive Gra...,1,True,True
5,invoice.seller_name,0,0,KMOZ 92.3 The Moose,KMOZ 92.3 The Moose,1,True,True
6,payment_instructions.payment_terms,0,0,30 Days,30 Days,1,True,True
7,subtotal.total,0,0,$806.00,$806.00,1,True,True


In [90]:
from app.extraction.bert.schema import InvoiceLabelSchema

schema = InvoiceLabelSchema()
supported_fields = set(schema.field_to_entity)

In [91]:
def prepare_records(df, require_text_equality=False):
    records = []
    rejected = []

    for row_index, row in df.iterrows():
        spans = find_spans(row)
        comparison = compare_spans(row, spans)

        # Keep only fields supported by the current model.
        spans = [
            span for span in spans
            if span["label"] in supported_fields
        ]
        comparison = comparison.loc[
            comparison["field"].isin(supported_fields)
        ]

        issues = []

        if comparison.empty:
            issues.append({"status": "no_supported_annotations"})

        for item in comparison.to_dict("records"):
            if not item["matched"]:
                issues.append({
                    **item,
                    "status": "missing_span",
                })
            elif require_text_equality and not item["normalized_equal"]:
                issues.append({
                    **item,
                    "status": "text_mismatch",
                })

        # Remove exact duplicate spans before checking overlaps.
        spans = list({
            (
                span["start"],
                span["end"],
                span["label"],
                span["page"],
            ): span
            for span in spans
        }.values())

        spans.sort(key=lambda span: (span["start"], span["end"]))

        for span in spans:
            if row["ocr_text"][span["start"]:span["end"]] != span["text"]:
                issues.append({
                    "field": span["label"],
                    "status": "invalid_offsets",
                })

        for left, right in zip(spans, spans[1:]):
            if right["start"] < left["end"]:
                issues.append({
                    "status": "overlap",
                    "field_a": left["label"],
                    "field_b": right["label"],
                })

        if issues:
            rejected.append({
                "row_index": row_index,
                "document_id": row["document_id"],
                "issues": issues,
            })
        else:
            records.append({
                "row_index": row_index,
                "document_id": row["document_id"],
                "text": row["ocr_text"],
                "entities": spans,
                "ignored_regions": [],
                "text_differences": comparison.loc[
                    ~comparison["normalized_equal"]
                ].to_dict("records"),
            })

    return records, rejected

In [92]:
records, rejected = prepare_records(df)

print("Accepted:", len(records))
print("Rejected:", len(rejected))

Accepted: 782
Rejected: 4898


In [93]:
rejection_reasons = pd.DataFrame([
    {
        "document_id": record["document_id"],
        **issue,
    }
    for record in rejected
    for issue in record["issues"]
])

In [95]:
rejection_reasons["status"].value_counts()

status
overlap         10716
missing_span      318
Name: count, dtype: int64

In [98]:
rejection_reasons.groupby("status")["document_id"].nunique().rename("documents")

status
missing_span     256
overlap         4836
Name: documents, dtype: int64

## 1. Select target fields. Include only fields you intend to train.

In [99]:
from collections import Counter

SECTIONS = ("invoice", "payment_instructions", "subtotal")

field_counts = Counter()

for data in df["json_data"]:
    document_fields = set()

    for section in SECTIONS:
        for name, field in data.get(section, {}).items():
            if field is None:
                continue

            value = field.get("value")

            if isinstance(value, str) and value.strip():
                document_fields.add(f"{section}.{name}")

    # Count each field once per document, regardless of repeated boxes.
    field_counts.update(document_fields)

coverage_df = pd.DataFrame(
    field_counts.items(),
    columns=["field", "documents"],
)

coverage_df["coverage"] = coverage_df["documents"] / len(df)
coverage_df.sort_values("documents", ascending=False)

,field,documents,coverage
0,invoice.client_name,5488,0.966197
4,subtotal.total,5337,0.939613
5,invoice.invoice_date,5013,0.882570
7,invoice.seller_name,4968,0.874648
8,invoice.client_address,4743,0.835035
6,invoice.invoice_number,4711,0.829401
3,invoice.seller_address,4577,0.805810
1,payment_instructions.payment_terms,2041,0.359331
10,invoice.due_date,742,0.130634
9,subtotal.tax,610,0.107394


In [ ]:
TARGET_FIELDS = {
    "invoice.client_name",
    "invoice.client_address",
    "invoice.seller_name",
    "invoice.seller_address",
    "invoice.invoice_number",
    "invoice.invoice_date",
    "invoice.due_date",
    "payment_instructions.payment_terms",
    "payment_instructions.account_number",
    "subtotal.tax",
    "subtotal.total",
}

In [101]:
SECTIONS = ("invoice", "payment_instructions", "subtotal")


def find_spans(row):
    ocr_text = row["ocr_text"]
    text_with_boxes = row["text_with_boxes"]
    spans = []

    for section in SECTIONS:
        for name, field in row["json_data"].get(section, {}).items():
            if field is None:
                continue

            path = f"{section}.{name}"
            if path not in TARGET_FIELDS:
                continue

            for occurrence, (bbox, page) in enumerate(
                zip(field["bbox"], field["pages"], strict=True)
            ):
                previous_index = None

                for word_index, word in enumerate(text_with_boxes):
                    if not word_inside_box(word, bbox, page):
                        continue

                    if (
                        previous_index is not None
                        and word_index == previous_index + 1
                    ):
                        spans[-1]["end"] = int(word["end"])
                    else:
                        spans.append({
                            "start": int(word["start"]),
                            "end": int(word["end"]),
                            "label": path,
                            "page": int(page),
                            "occurrence": occurrence,
                        })

                    previous_index = word_index

    spans.sort(key=lambda span: span["start"])

    for span in spans:
        span["text"] = ocr_text[span["start"]:span["end"]]

    return spans

## 2. Assign candidate labels to OCR words using boxes.

In [102]:
def assign_candidate_labels(row):
    words = row["text_with_boxes"]

    # Preserve the original word order and metadata.
    word_candidates = [
        {
            **word,
            "ocr_index": index,
            "candidates": [],
        }
        for index, word in enumerate(words)
    ]

    annotations = []

    for section in SECTIONS:
        for name, field in row["json_data"].get(section, {}).items():
            path = f"{section}.{name}"

            if field is None or path not in TARGET_FIELDS:
                continue

            for occurrence, (bbox, page) in enumerate(
                zip(field["bbox"], field["pages"], strict=True)
            ):
                annotation_id = len(annotations)
                matched_words = 0

                for word_index, word in enumerate(words):
                    if not word_inside_box(word, bbox, page):
                        continue

                    word_candidates[word_index]["candidates"].append({
                        "label": path,
                        "occurrence": occurrence,
                        "annotation_id": annotation_id,
                    })

                    matched_words += 1

                annotations.append({
                    "annotation_id": annotation_id,
                    "field": path,
                    "occurrence": occurrence,
                    "page": int(page),
                    "expected": field["source_values"][occurrence],
                    "matched_words": matched_words,
                })

    return word_candidates, annotations

In [103]:
row = df.iloc[1]

word_candidates, annotations = assign_candidate_labels(row)

In [104]:
candidates_df = pd.DataFrame(word_candidates)

candidates_df.loc[
    candidates_df["candidates"].map(bool),
    ["ocr_index", "page", "start", "end", "text", "candidates"],
]

,ocr_index,page,start,end,text,candidates
3,3,0,17,21,KMOZ,"[{'label': 'invoice.seller_address', 'occurren..."
4,4,0,22,26,92.3,"[{'label': 'invoice.seller_address', 'occurren..."
5,5,0,27,30,The,"[{'label': 'invoice.seller_address', 'occurren..."
6,6,0,31,36,Moose,"[{'label': 'invoice.seller_address', 'occurren..."
7,7,0,37,41,1360,"[{'label': 'invoice.seller_address', 'occurren..."
8,8,0,42,44,E.,"[{'label': 'invoice.seller_address', 'occurren..."
9,9,0,45,53,Sherwood,"[{'label': 'invoice.seller_address', 'occurren..."
10,10,0,54,59,Drive,"[{'label': 'invoice.seller_address', 'occurren..."
11,11,0,60,65,Grand,"[{'label': 'invoice.seller_address', 'occurren..."
12,12,0,66,75,"Junction,","[{'label': 'invoice.seller_address', 'occurren..."


In [105]:
annotations_df = pd.DataFrame(annotations)
annotations_df

,annotation_id,field,occurrence,page,expected,matched_words
0,0,invoice.client_address,0,1,Mesa County Republicans\n920 Visa Ct.\nGrand J...,10
1,1,invoice.client_address,1,0,Mesa County Republicans\n920 Visa Ct.\nGrand J...,10
2,2,invoice.client_name,0,0,Mesa County Republicans,3
3,3,invoice.client_name,1,1,Mesa County Republicans,3
4,4,invoice.invoice_date,0,0,10/31/2020,1
5,5,invoice.invoice_date,1,1,10/31/2020,1
6,6,invoice.invoice_number,0,0,17685-2,1
7,7,invoice.invoice_number,1,1,17685-2,1
8,8,invoice.seller_address,0,1,KMOZ 92.3 The Moose\n1360 E. Sherwood Drive\nG...,12
9,9,invoice.seller_address,1,0,KMOZ 92.3 The Moose\n1360 E. Sherwood Drive\nG...,12


In [106]:
conflict_mask = candidates_df["candidates"].map(
    lambda candidates: len({
        candidate["label"]
        for candidate in candidates
    }) > 1
)

candidates_df.loc[
    conflict_mask,
    ["ocr_index", "page", "text", "candidates"],
]

,ocr_index,page,text,candidates
3,3,0,KMOZ,"[{'label': 'invoice.seller_address', 'occurren..."
4,4,0,92.3,"[{'label': 'invoice.seller_address', 'occurren..."
5,5,0,The,"[{'label': 'invoice.seller_address', 'occurren..."
6,6,0,Moose,"[{'label': 'invoice.seller_address', 'occurren..."
25,25,0,Mesa,"[{'label': 'invoice.client_address', 'occurren..."
26,26,0,County,"[{'label': 'invoice.client_address', 'occurren..."
27,27,0,Republicans,"[{'label': 'invoice.client_address', 'occurren..."
263,263,1,KMOZ,"[{'label': 'invoice.seller_address', 'occurren..."
264,264,1,92.3,"[{'label': 'invoice.seller_address', 'occurren..."
265,265,1,The,"[{'label': 'invoice.seller_address', 'occurren..."


## 3. Resolve name/address nesting. Give the name label priority over the corresponding address label.

In [107]:
NAME_TO_ADDRESS = {
    "invoice.client_name": "invoice.client_address",
    "invoice.seller_name": "invoice.seller_address",
}


def resolve_name_address_nesting(word_candidates):
    resolved_words = []

    for word in word_candidates:
        labels = {
            candidate["label"]
            for candidate in word["candidates"]
        }

        excluded_addresses = {
            address
            for name, address in NAME_TO_ADDRESS.items()
            if name in labels
        }

        resolved_words.append({
            **word,
            "candidates": [
                candidate.copy()
                for candidate in word["candidates"]
                if candidate["label"] not in excluded_addresses
            ],
        })

    return resolved_words

In [108]:
resolved_words = resolve_name_address_nesting(word_candidates)
resolved_df = pd.DataFrame(resolved_words)

In [109]:
remaining_conflict_mask = resolved_df["candidates"].map(
    lambda candidates: len({
        candidate["label"]
        for candidate in candidates
    }) > 1
)

resolved_df.loc[
    remaining_conflict_mask,
    ["ocr_index", "page", "text", "candidates"],
]

,ocr_index,page,text,candidates


In [110]:
print("Before:", int(conflict_mask.sum()))
print("After:", int(remaining_conflict_mask.sum()))

Before: 14
After: 0


## 4. Deduplicate identical labels on the same word.

In [111]:
def deduplicate_candidate_labels(resolved_words):
    deduplicated_words = []

    for word in resolved_words:
        by_label = {}

        for candidate in word["candidates"]:
            label = candidate["label"]

            grouped = by_label.setdefault(
                label,
                {
                    "label": label,
                    "sources": [],
                },
            )

            source = {
                "annotation_id": candidate["annotation_id"],
                "occurrence": candidate["occurrence"],
            }

            if source not in grouped["sources"]:
                grouped["sources"].append(source)

        deduplicated_words.append({
            **word,
            "candidates": list(by_label.values()),
        })

    return deduplicated_words

In [112]:
deduplicated_words = deduplicate_candidate_labels(resolved_words)
deduplicated_df = pd.DataFrame(deduplicated_words)

deduplicated_df.loc[
    deduplicated_df["candidates"].map(bool),
    ["ocr_index", "page", "text", "candidates"],
]

,ocr_index,page,text,candidates
3,3,0,KMOZ,"[{'label': 'invoice.seller_name', 'sources': [..."
4,4,0,92.3,"[{'label': 'invoice.seller_name', 'sources': [..."
5,5,0,The,"[{'label': 'invoice.seller_name', 'sources': [..."
6,6,0,Moose,"[{'label': 'invoice.seller_name', 'sources': [..."
7,7,0,1360,"[{'label': 'invoice.seller_address', 'sources'..."
8,8,0,E.,"[{'label': 'invoice.seller_address', 'sources'..."
9,9,0,Sherwood,"[{'label': 'invoice.seller_address', 'sources'..."
10,10,0,Drive,"[{'label': 'invoice.seller_address', 'sources'..."
11,11,0,Grand,"[{'label': 'invoice.seller_address', 'sources'..."
12,12,0,"Junction,","[{'label': 'invoice.seller_address', 'sources'..."


In [113]:
deduplicated_df.loc[
    deduplicated_df["candidates"].map(len) > 1,
    ["ocr_index", "page", "text", "candidates"],
]

,ocr_index,page,text,candidates


## 5. Build spans from the resolved word assignments.

In [114]:
def build_spans(ocr_text, deduplicated_words):
    spans = []
    ignored_regions = []
    previous_key = None
    previous_index = None

    for word in deduplicated_words:
        candidates = word["candidates"]

        if len(candidates) != 1:
            if len(candidates) > 1:
                ignored_regions.append({
                    "start": int(word["start"]),
                    "end": int(word["end"]),
                    "page": int(word["page"]),
                })

            previous_key = None
            previous_index = None
            continue

        candidate = candidates[0]

        source_key = tuple(sorted(
            (source["annotation_id"], source["occurrence"])
            for source in candidate["sources"]
        ))

        key = (
            candidate["label"],
            int(word["page"]),
            source_key,
        )

        start = int(word["start"])
        end = int(word["end"])

        if (
            key == previous_key
            and word["ocr_index"] == previous_index + 1
            and not ocr_text[spans[-1]["end"]:start].strip()
        ):
            spans[-1]["end"] = end
        else:
            spans.append({
                "start": start,
                "end": end,
                "label": candidate["label"],
                "page": int(word["page"]),
                "sources": [
                    source.copy()
                    for source in candidate["sources"]
                ],
            })

        previous_key = key
        previous_index = word["ocr_index"]

    for span in spans:
        span["text"] = ocr_text[span["start"]:span["end"]]

    return spans, ignored_regions

In [115]:
spans, ignored_regions = build_spans(
    row["ocr_text"],
    deduplicated_words,
)

spans_df = pd.DataFrame(spans)
spans_df


,start,end,label,page,sources,text
0,17,36,invoice.seller_name,0,"[{'annotation_id': 10, 'occurrence': 0}]",KMOZ 92.3 The Moose
1,37,84,invoice.seller_address,0,"[{'annotation_id': 9, 'occurrence': 1}]","1360 E. Sherwood Drive Grand Junction, CO 81501"
2,119,126,invoice.invoice_number,0,"[{'annotation_id': 6, 'occurrence': 0}]",17685-2
3,127,137,invoice.invoice_date,0,"[{'annotation_id': 4, 'occurrence': 0}]",10/31/2020
4,153,160,subtotal.total,0,"[{'annotation_id': 12, 'occurrence': 0}]",$806.00
5,161,184,invoice.client_name,0,"[{'annotation_id': 2, 'occurrence': 0}]",Mesa County Republicans
6,208,220,invoice.client_address,0,"[{'annotation_id': 1, 'occurrence': 1}]",920 Visa Ct.
7,244,268,invoice.client_address,0,"[{'annotation_id': 1, 'occurrence': 1}]","Grand Junction, CO 81506"
8,1624,1643,invoice.seller_name,1,"[{'annotation_id': 11, 'occurrence': 1}]",KMOZ 92.3 The Moose
9,1644,1691,invoice.seller_address,1,"[{'annotation_id': 8, 'occurrence': 0}]","1360 E. Sherwood Drive Grand Junction, CO 81501"


In [116]:
pd.DataFrame(
    ignored_regions,
    columns=["start", "end", "page"],
)

,start,end,page


In [117]:
assert all(
    row["ocr_text"][span["start"]:span["end"]] == span["text"]
    for span in spans
)

assert all(
    left["end"] <= right["start"]
    for left, right in zip(spans, spans[1:])
)

## 6. Report only remaining ambiguities as overlap issues.

In [118]:
from collections import Counter


def find_remaining_issues(deduplicated_words, annotations):
    issues = []
    supervised_counts = Counter()

    if not annotations:
        return [{"status": "no_selected_annotations"}]

    for word in deduplicated_words:
        candidates = word["candidates"]

        if len(candidates) > 1:
            issues.append({
                "status": "overlap",
                "page": int(word["page"]),
                "start": int(word["start"]),
                "end": int(word["end"]),
                "text": word["text"],
                "labels": [
                    candidate["label"]
                    for candidate in candidates
                ],
            })

        elif len(candidates) == 1:
            # Count this word once for each supporting annotation.
            supervised_counts.update({
                source["annotation_id"]
                for source in candidates[0]["sources"]
            })

    for annotation in annotations:
        annotation_id = annotation["annotation_id"]

        if annotation["matched_words"] == 0:
            status = "missing_span"
        elif supervised_counts[annotation_id] == 0:
            status = "no_supervised_words"
        else:
            continue

        issues.append({
            **annotation,
            "status": status,
            "supervised_words": supervised_counts[annotation_id],
        })

    return issues

In [119]:
issues = find_remaining_issues(
    deduplicated_words,
    annotations,
)

pd.DataFrame(issues)

""


## 7. Combine all together

In [120]:
def prepare_docile_records(df):
    records = []
    rejected = []

    for row_index, row in df.iterrows():
        word_candidates, annotations = assign_candidate_labels(row)

        resolved_words = resolve_name_address_nesting(word_candidates)
        deduplicated_words = deduplicate_candidate_labels(resolved_words)

        spans, ignored_regions = build_spans(
            row["ocr_text"],
            deduplicated_words,
        )

        issues = find_remaining_issues(
            deduplicated_words,
            annotations,
        )

        for span in spans:
            start, end = span["start"], span["end"]

            if not (
                0 <= start < end <= len(row["ocr_text"])
                and row["ocr_text"][start:end] == span["text"]
            ):
                issues.append({
                    "status": "invalid_offsets",
                    "field": span["label"],
                    "start": start,
                    "end": end,
                })

        if issues:
            rejected.append({
                "row_index": row_index,
                "document_id": row["document_id"],
                "issues": issues,
            })
        else:
            records.append({
                "row_index": row_index,
                "document_id": row["document_id"],
                "text": row["ocr_text"],
                "entities": spans,
                "ignored_regions": ignored_regions,
            })

    return records, rejected

In [121]:
records, rejected = prepare_docile_records(df)

print("Accepted:", len(records))
print("Rejected:", len(rejected))

Accepted: 5403
Rejected: 277


In [122]:
rejection_reasons = pd.DataFrame([
    {
        "document_id": record["document_id"],
        **issue,
    }
    for record in rejected
    for issue in record["issues"]
])

rejection_reasons.groupby("status").agg(
    issues=("document_id", "size"),
    documents=("document_id", "nunique"),
)


,issues,documents
status,,
missing_span,318,256
no_supervised_words,23,13
overlap,38,15


In [5]:
token_debug_df = pd.read_csv(
    "../data/interim/bert_diagnostics/epoch_016/"
    "tokens_b66a1955cd674319890c4a1a_window_000.csv",
    keep_default_na=False,
)

token_debug_df.loc[
    (token_debug_df["start"] < 834)
    & (token_debug_df["end"] > 787),
    [
        "token_index",
        "token",
        "ocr_text",
        "start",
        "end",
        "actual",
        "predicted",
        "predicted_probability",
        "actual_probability",
    ],
]


,token_index,token,ocr_text,start,end,actual,predicted,predicted_probability,actual_probability
274,274,am,AM,787,789,O,O,0.998138,0.9981383085250854
275,275,spots,Spots,790,795,O,O,0.998967,0.9989668130874634
276,276,23,23,796,798,O,O,0.997849,0.9978491067886353
277,277,$,$,799,800,O,O,0.966450,0.9664497375488281
278,278,71,71,800,802,O,O,0.976839,0.9768391251564026
279,279,##3,3,802,803,O,O,0.967509,0.9675090312957764
280,280,.,.,803,804,O,O,0.969169,0.9691693186759949
281,281,00,00,804,806,O,O,0.957524,0.9575238823890686
282,282,$,$,807,808,B-TOTAL,O,0.951558,0.0345686711370945
283,283,71,71,808,810,I-TOTAL,O,0.950811,0.0375094935297966


In [7]:
import json
import pandas as pd

from app.extraction.bert.spans import word_inside_box

document_id = "b50f9762f15b45b4b652af61"

false_positives = pd.read_csv(
    "../data/interim/bert_diagnostics/epoch_016/false_positive.csv",
    keep_default_na=False,
)

prediction = false_positives.loc[
    (false_positives["document_id"] == document_id)
    & (false_positives["label"] == "TOTAL")
].iloc[0]

# Use the prepared dataset from this training run.
source_df = pd.read_parquet("../docile_df.parquet")

record = source_df.loc[
    source_df["document_id"] == document_id
].iloc[0]

json_data = record["json_data"]
if isinstance(json_data, str):
    json_data = json.loads(json_data)

words = record["text_with_boxes"]
if isinstance(words, str):
    words = json.loads(words)

start = int(prediction["start"])
end = int(prediction["end"])

In [ ]:
predicted_words = [
    word
    for word in words
    if start < word["end"] and end > word["start"]
]

pw_df = pd.DataFrame(predicted_words)[
    ["text", "page", "start", "end", "bbox"]
]
pw_df.to_csv('./predicted_words.csv')

In [10]:
amount_fields = (
    ("subtotal", "total"),
    ("other_fields", "amount_due"),
    ("other_fields", "amount_total_net"),
    ("other_fields", "amount_paid"),
    ("other_fields", "tax_detail_net"),
    ("other_fields", "tax_detail_gross"),
)

rows = []

for section, name in amount_fields:
    field = json_data.get(section, {}).get(name)

    if field is None:
        continue

    for occurrence, (bbox, page) in enumerate(
        zip(field["bbox"], field["pages"], strict=True)
    ):
        matching_words = [
            word
            for word in predicted_words
            if word_inside_box(word, bbox, page)
        ]

        rows.append({
            "field": f"{section}.{name}",
            "occurrence": occurrence,
            "page": int(page),
            "expected": field["source_values"][occurrence],
            "predicted_text": prediction["text"],
            "matching_words": len(matching_words),
            "at_prediction_location": bool(matching_words),
        })

amount_comparison_df = pd.DataFrame(rows)

amount_comparison_df

,field,occurrence,page,expected,predicted_text,matching_words,at_prediction_location
0,subtotal.total,0,1,848.00,$848.00,0,False
1,other_fields.amount_due,0,0,$848.00,$848.00,1,True
2,other_fields.amount_due,1,1,$848.00,$848.00,0,False
3,other_fields.amount_total_net,0,1,848.00,$848.00,0,False


In [14]:
document_id

'b50f9762f15b45b4b652af61'

In [18]:
import json
import pandas as pd

document_id = "657e3c28c1d74e698a0cd884"

errors = pd.read_csv(
    "../data/interim/bert_diagnostics/epoch_016/false_positive.csv",
    keep_default_na=False,
)

candidates = errors.loc[
    (errors["document_id"] == document_id)
    & (errors["label"] == "SELLER_ADDRESS")
].copy()

candidates = candidates.loc[
    candidates["overlapping_entities"].map(
        lambda value: any(
            entity["label"] == "CLIENT_ADDRESS"
            for entity in json.loads(value)
        )
    )
]

assert not candidates.empty

prediction = candidates.iloc[0]
prediction[["text", "start", "end", "overlapping_entities"]].to_dict()

{'text': '919 CATHARINE STREET PHILADELPHIA, PA 19147',
 'start': 1991,
 'end': 2034,
 'overlapping_entities': '[{"label": "CLIENT_ADDRESS", "start": 1991, "end": 2034, "text": "919 CATHARINE STREET PHILADELPHIA, PA 19147"}]'}

In [20]:
import json
import pandas as pd

case_document_id = "657e3c28c1d74e698a0cd884"

case_rows = source_df.loc[
    source_df["document_id"] == case_document_id
]
assert len(case_rows) == 1

case_record = case_rows.iloc[0]
case_text = case_record["ocr_text"]

case_words = case_record["text_with_boxes"]
if isinstance(case_words, str):
    case_words = json.loads(case_words)

case_words_df = pd.DataFrame(list(case_words))
case_words_df["ocr_position"] = range(len(case_words_df))

# Check that the word metadata belongs to this OCR string.
assert all(
    case_text[int(word["start"]):int(word["end"])] == word["text"]
    for word in case_words
)

case_start, case_end = 1991, 2034

assert case_text[case_start:case_end] == (
    "919 CATHARINE STREET PHILADELPHIA, PA 19147"
)

case_selected = case_words_df.loc[
    (case_words_df["start"] < case_end)
    & (case_words_df["end"] > case_start)
]

assert case_selected["page"].nunique() == 1
case_page = int(case_selected["page"].iloc[0])

case_context_df = case_words_df.loc[
    (case_words_df["page"] == case_page)
    & (case_words_df["end"] > case_start - 150)
    & (case_words_df["start"] < case_end + 100)
]

case_context_df[
    [
        "ocr_position",
        "text",
        "page",
        "start",
        "end",
        "block_index",
        "line_index",
        "word_index",
    ]
].to_dict(orient='records')

[{'ocr_position': 311,
  'text': 'Phone:',
  'page': 1,
  'start': 1839,
  'end': 1845,
  'block_index': 5,
  'line_index': 0,
  'word_index': 4},
 {'ocr_position': 312,
  'text': '217-324-5921;',
  'page': 1,
  'start': 1846,
  'end': 1859,
  'block_index': 5,
  'line_index': 0,
  'word_index': 5},
 {'ocr_position': 313,
  'text': '217-532-2066;',
  'page': 1,
  'start': 1860,
  'end': 1873,
  'block_index': 5,
  'line_index': 0,
  'word_index': 6},
 {'ocr_position': 314,
  'text': 'CONTRACT:',
  'page': 1,
  'start': 1874,
  'end': 1883,
  'block_index': 1,
  'line_index': 5,
  'word_index': 0},
 {'ocr_position': 315,
  'text': '1153501',
  'page': 1,
  'start': 1884,
  'end': 1891,
  'block_index': 1,
  'line_index': 5,
  'word_index': 1},
 {'ocr_position': 316,
  'text': '217-854-2312',
  'page': 1,
  'start': 1892,
  'end': 1904,
  'block_index': 0,
  'line_index': 4,
  'word_index': 0},
 {'ocr_position': 317,
  'text': 'FAX:',
  'page': 1,
  'start': 1905,
  'end': 1909,
  'block

In [21]:
import json
from pathlib import Path

import pandas as pd

diagnostic_dir = Path(
    "../data/interim/docile_diagnostics_20_epochs/epoch_019"
)

errors_df = pd.read_csv(
    diagnostic_dir / "entity_errors.csv",
    keep_default_na=False,
)

false_positives = errors_df.loc[
    errors_df["status"].eq("false_positive")
].copy()

false_positives["references"] = (
    false_positives["overlapping_entities"].map(json.loads)
)

In [22]:
rows = []

for prediction_id, prediction in false_positives.iterrows():
    for gold in prediction["references"]:
        if gold["label"] != prediction["label"]:
            continue

        predicted_start = int(prediction["start"])
        predicted_end = int(prediction["end"])
        gold_start = int(gold["start"])
        gold_end = int(gold["end"])

        if predicted_start == gold_start and predicted_end == gold_end:
            boundary_type = "same_boundaries"
        elif predicted_start >= gold_start and predicted_end <= gold_end:
            boundary_type = "partial"
        elif predicted_start <= gold_start and predicted_end >= gold_end:
            boundary_type = "expanded"
        else:
            boundary_type = "shifted"

        rows.append({
            "prediction_id": prediction_id,
            "document_id": prediction["document_id"],
            "window_index": int(prediction["window_index"]),
            "label": prediction["label"],
            "predicted_text": prediction["text"],
            "predicted_start": predicted_start,
            "predicted_end": predicted_end,
            "gold_text": gold["text"],
            "gold_start": gold_start,
            "gold_end": gold_end,
            "start_difference": predicted_start - gold_start,
            "end_difference": predicted_end - gold_end,
            "boundary_type": boundary_type,
            "context": prediction["context"],
        })

boundary_df = pd.DataFrame(rows)

boundary_df.groupby(
    ["label", "boundary_type"]
).size().unstack(fill_value=0)

boundary_type,expanded,partial,shifted
label,,,
ACCOUNT_NUMBER,0,1,1
AMOUNT_DUE,1,12,0
CLIENT_ADDRESS,17,94,2
CLIENT_NAME,35,53,4
DUE_DATE,0,3,0
INVOICE_DATE,2,12,0
INVOICE_NUMBER,13,22,0
PAYMENT_TERMS,14,16,1
SELLER_ADDRESS,20,78,1


In [23]:
gold_key = [
    "document_id",
    "window_index",
    "label",
    "gold_start",
    "gold_end",
]

boundary_df["predictions_per_gold"] = (
    boundary_df.groupby(gold_key)["prediction_id"]
    .transform("nunique")
)

boundary_df["gold_entities_per_prediction"] = (
    boundary_df.groupby("prediction_id")["gold_start"]
    .transform("size")
)

boundary_df["possible_split"] = (
    boundary_df["predictions_per_gold"] > 1
)

boundary_df["possible_merge"] = (
    boundary_df["gold_entities_per_prediction"] > 1
)

boundary_df[
    ["possible_split", "possible_merge"]
].sum()


possible_split    170
possible_merge     16
dtype: int64

In [28]:
columns = [
    "document_id",
    "window_index",
    "label",
    "gold_text",
    "predicted_text",
    "start_difference",
    "end_difference",
    "possible_split",
    "possible_merge",
]

boundary_df.loc[
    boundary_df["label"].eq("TOTAL"),
    columns,
].to_dict(orient='records')

[{'document_id': 'ff34fb7572224f22ba00f01e',
  'window_index': 0,
  'label': 'TOTAL',
  'gold_text': '$15,000.',
  'predicted_text': '$15,000. F',
  'start_difference': 0,
  'end_difference': 2,
  'possible_split': False,
  'possible_merge': False},
 {'document_id': '50678ecc66eb4465ba98b2b6',
  'window_index': 0,
  'label': 'TOTAL',
  'gold_text': '$137,756.25',
  'predicted_text': '$137,756.25 -',
  'start_difference': 0,
  'end_difference': 2,
  'possible_split': False,
  'possible_merge': False},
 {'document_id': 'a8749aaacd0c48e1ab7a8abb',
  'window_index': 1,
  'label': 'TOTAL',
  'gold_text': '3,893.17',
  'predicted_text': '$ 3,893.17',
  'start_difference': -2,
  'end_difference': 0,
  'possible_split': False,
  'possible_merge': False},
 {'document_id': '3061c709851a4e76b07faae3',
  'window_index': 0,
  'label': 'TOTAL',
  'gold_text': '7,465.25',
  'predicted_text': ',465.25',
  'start_difference': 1,
  'end_difference': 0,
  'possible_split': False,
  'possible_merge': Fals

In [29]:
columns = [
    "document_id",
    "window_index",
    "label",
    "gold_text",
    "predicted_text",
    "start_difference",
    "end_difference",
    "possible_split",
    "possible_merge",
]

boundary_df.loc[
    boundary_df["label"].eq("SELLER_ADDRESS"),
    columns,
].to_dict(orient='records')

[{'document_id': '00aa98164d264f4e924f55a9',
  'window_index': 0,
  'label': 'SELLER_ADDRESS',
  'gold_text': '1333 Lakeside Ave',
  'predicted_text': '3 Lakeside Ave',
  'start_difference': 3,
  'end_difference': 0,
  'possible_split': False,
  'possible_merge': False},
 {'document_id': 'a8749aaacd0c48e1ab7a8abb',
  'window_index': 0,
  'label': 'SELLER_ADDRESS',
  'gold_text': 'MOZES HOUSE, 74.76 ROTHSCHID BLVD. (4th FL.) 65783 TEL-AVIV',
  'predicted_text': 'MOZES HOUSE, 74.76 ROTHSCHID BLVD. (4th FL.) 65783',
  'start_difference': 0,
  'end_difference': -9,
  'possible_split': False,
  'possible_merge': False},
 {'document_id': 'a8749aaacd0c48e1ab7a8abb',
  'window_index': 1,
  'label': 'SELLER_ADDRESS',
  'gold_text': 'MOZES HOUSE, 74-76 ROTHSCHILD BLVD. (4th FLOOR) TEL-AVIV 65 785',
  'predicted_text': 'ZES HOUSE, 74-76 ROTHSCHILD BLVD. (4th FLOOR) TEL-AVIV 65',
  'start_difference': 2,
  'end_difference': -4,
  'possible_split': False,
  'possible_merge': False},
 {'document_id'

In [30]:
columns = [
    "document_id",
    "window_index",
    "label",
    "gold_text",
    "predicted_text",
    "start_difference",
    "end_difference",
    "possible_split",
    "possible_merge",
]

boundary_df.loc[
    boundary_df["label"].eq("CLIENT_ADDRESS"),
    columns,
].to_dict(orient='records')

[{'document_id': 'fc336e3fbf8a4c69aea2637d',
  'window_index': 0,
  'label': 'CLIENT_ADDRESS',
  'gold_text': 'Attn: Morgan Lilich',
  'predicted_text': 'Attn: Morgan Lilich House',
  'start_difference': 0,
  'end_difference': 6,
  'possible_split': False,
  'possible_merge': False},
 {'document_id': 'fc336e3fbf8a4c69aea2637d',
  'window_index': 0,
  'label': 'CLIENT_ADDRESS',
  'gold_text': '131 N. Santa Fe',
  'predicted_text': 'N. Santa Fe',
  'start_difference': 4,
  'end_difference': 0,
  'possible_split': False,
  'possible_merge': False},
 {'document_id': 'fc336e3fbf8a4c69aea2637d',
  'window_index': 0,
  'label': 'CLIENT_ADDRESS',
  'gold_text': 'Suite3 3',
  'predicted_text': 'Suite3',
  'start_difference': 0,
  'end_difference': -2,
  'possible_split': False,
  'possible_merge': False},
 {'document_id': 'fc336e3fbf8a4c69aea2637d',
  'window_index': 0,
  'label': 'CLIENT_ADDRESS',
  'gold_text': 'Salina, KS 67401',
  'predicted_text': '67',
  'start_difference': 11,
  'end_dif

In [31]:
columns = [
    "document_id",
    "window_index",
    "label",
    "gold_text",
    "predicted_text",
    "start_difference",
    "end_difference",
    "possible_split",
    "possible_merge",
]

boundary_df.loc[
    boundary_df["label"].eq("PAYMENT_TERMS"),
    columns,
].to_dict(orient='records')

[{'document_id': '6ebbc013ec674df8958604a8',
  'window_index': 0,
  'label': 'PAYMENT_TERMS',
  'gold_text': '30 days',
  'predicted_text': '30 days from invoice date.',
  'start_difference': 0,
  'end_difference': 19,
  'possible_split': False,
  'possible_merge': False},
 {'document_id': '12eb1b2292544172ba5e59e6',
  'window_index': 1,
  'label': 'PAYMENT_TERMS',
  'gold_text': 'Payment is due 30 days from receipt of invoice.',
  'predicted_text': 'Payment',
  'start_difference': 0,
  'end_difference': -40,
  'possible_split': True,
  'possible_merge': False},
 {'document_id': '12eb1b2292544172ba5e59e6',
  'window_index': 1,
  'label': 'PAYMENT_TERMS',
  'gold_text': 'Payment is due 30 days from receipt of invoice.',
  'predicted_text': 'is due 30 days from receipt of invoice.',
  'start_difference': 8,
  'end_difference': 0,
  'possible_split': True,
  'possible_merge': False},
 {'document_id': 'c411c13e968c43b59835e5fa',
  'window_index': 0,
  'label': 'PAYMENT_TERMS',
  'gold_text

In [33]:
example = boundary_df.loc[
    boundary_df["label"].eq("TOTAL")
].iloc[0]

example[columns + ["context"]]

document_id                                  ff34fb7572224f22ba00f01e
window_index                                                        0
label                                                           TOTAL
gold_text                                                    $15,000.
predicted_text                                             $15,000. F
start_difference                                                    0
end_difference                                                      2
possible_split                                                  False
possible_merge                                                  False
context             29 SE SURVEYS JOBI # DATE 1903 DESCRIPTION 1/1...
Name: 1, dtype: object

In [34]:
source_df = pd.read_parquet("../docile_df.parquet")

document = source_df.loc[
    source_df["document_id"].eq(example["document_id"])
].iloc[0]

text = document["ocr_text"]
words = document["text_with_boxes"]

if isinstance(words, str):
    words = json.loads(words)

assert text[
    example["gold_start"]:example["gold_end"]
] == example["gold_text"]

assert text[
    example["predicted_start"]:example["predicted_end"]
] == example["predicted_text"]

In [35]:
nearby_words = pd.DataFrame([
    {
        "text": word["text"],
        "start": int(word["start"]),
        "end": int(word["end"]),
        "start_cut": (
            word["start"]
            < example["predicted_start"]
            < word["end"]
        ),
        "end_cut": (
            word["start"]
            < example["predicted_end"]
            < word["end"]
        ),
    }
    for word in words
    if word["end"] > example["gold_start"] - 40
    and word["start"] < example["gold_end"] + 40
])

nearby_words

,text,start,end,start_cut,end_cut
0,Smoker,250,256,False,False
1,Profile,257,264,False,False
2,Study,265,270,False,False
3,AE,271,273,False,False
4,Initial,274,281,False,False
5,Billing,282,289,False,False
6,"$15,000.",290,298,False,False
7,F,299,300,False,False


In [36]:
gold_key = [
    "document_id",
    "window_index",
    "label",
    "gold_start",
    "gold_end",
]

split_entities = (
    boundary_df.loc[boundary_df["possible_split"]]
    .drop_duplicates(gold_key)
)

merged_predictions = (
    boundary_df.loc[boundary_df["possible_merge"]]
    .drop_duplicates("prediction_id")
)

pd.Series({
    "gold_entities_with_multiple_predictions": len(split_entities),
    "predictions_overlapping_multiple_gold_entities": len(merged_predictions),
})

gold_entities_with_multiple_predictions           78
predictions_overlapping_multiple_gold_entities     8
dtype: int64

In [37]:
words_by_document = {}

for row in source_df[
    ["document_id", "text_with_boxes"]
].itertuples(index=False):
    words = row.text_with_boxes

    if isinstance(words, str):
        words = json.loads(words)

    words_by_document[row.document_id] = words


rows = []

# Count each false-positive prediction once.
predictions = boundary_df.drop_duplicates("prediction_id")

for prediction in predictions.itertuples(index=False):
    words = words_by_document[prediction.document_id]

    start_cut = any(
        word["start"] < prediction.predicted_start < word["end"]
        for word in words
    )
    end_cut = any(
        word["start"] < prediction.predicted_end < word["end"]
        for word in words
    )

    rows.append({
        "prediction_id": prediction.prediction_id,
        "label": prediction.label,
        "start_cut": start_cut,
        "end_cut": end_cut,
        "any_word_cut": start_cut or end_cut,
    })

word_cuts_df = pd.DataFrame(rows)

word_cut_summary = (
    word_cuts_df.groupby("label")
    .agg(
        predictions=("prediction_id", "size"),
        word_cuts=("any_word_cut", "sum"),
        word_cut_rate=("any_word_cut", "mean"),
    )
    .sort_values("word_cuts", ascending=False)
)

word_cut_summary

,predictions,word_cuts,word_cut_rate
label,,,
CLIENT_ADDRESS,111,30,0.270270
SELLER_ADDRESS,96,30,0.312500
SELLER_NAME,79,29,0.367089
TOTAL,46,27,0.586957
CLIENT_NAME,90,25,0.277778
INVOICE_NUMBER,35,19,0.542857
INVOICE_DATE,14,13,0.928571
AMOUNT_DUE,13,11,0.846154
PAYMENT_TERMS,31,6,0.193548
